# Agentic Alpha ML Throttle Search v1

Hypothesis: keep the validated multi-factor position budget as the base prior, and use supervised ML only as a throttle overlay. The model learns a 5-day forward return label, while the backtest uses daily returns and validation-only selection; 2026 remains a holdout report window.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

warnings.filterwarnings('ignore', message='X does not have valid feature names')

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import evaluate_ml_position_candidates, evaluate_weighted_position_candidates
from src.factor import (
    build_canary_sprint_frame,
    build_canary_trend_component,
    build_session_effect_frame,
    build_signed_overnight_component,
    build_tsmom_stop_component,
)
from soxl_backtest.reference import replay_soxlbacktest_reference

EXPERIMENT = 'agentic_alpha_ml_throttle_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
SOURCE_START = '2010-03-11'
DATA_START = '2019-01-02'
DATA_END = '2026-06-09'
SOURCE_END = '2026-06-23'
TRAIN_WINDOW = ('2019-01-02', '2024-12-31')
VALIDATION_WINDOW = ('2025-01-02', '2025-12-31')
TEST_WINDOW = ('2026-01-02', '2026-06-09')
LABEL_COL = 'label_return_5d'
DAILY_RETURN_COL = 'forward_return'
MAX_DRAWDOWN_FLOOR = -25.0
MAX_POSITION = 1.0
MIN_VALIDATION_TRADE_COUNT = 4
MIN_VALIDATION_AVG_ABS_POSITION = 0.10
MAX_VALIDATION_REVERSAL_RATIO = 0.40
COST_PER_TURNOVER = 0.0003

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = "\n".join([
    f"experiment: {EXPERIMENT}",
    "card_type: ml_throttle_position_search",
    "qlib:",
    f"  provider_uri: {PROVIDER_URI.as_posix()}",
    f"  recorder_uri: {RECORDER_URI.as_posix()}",
    "  market: all",
    "  region: us",
    "segments:",
    "  train:",
    f"    start: {TRAIN_WINDOW[0]}",
    f"    end: {TRAIN_WINDOW[1]}",
    "  validation:",
    f"    start: {VALIDATION_WINDOW[0]}",
    f"    end: {VALIDATION_WINDOW[1]}",
    "  test:",
    f"    start: {TEST_WINDOW[0]}",
    f"    end: {TEST_WINDOW[1]}",
    "ml_throttle_model:",
    f"  model_target_col: {LABEL_COL}",
    f"  return_col: {DAILY_RETURN_COL}",
    "  selection_segment: validation",
    "  report_segment: test",
    f"  max_position: {MAX_POSITION}",
    f"  min_validation_trade_count: {MIN_VALIDATION_TRADE_COUNT}",
    f"  min_validation_avg_abs_position: {MIN_VALIDATION_AVG_ABS_POSITION}",
    f"  max_validation_reversal_ratio: {MAX_VALIDATION_REVERSAL_RATIO}",
    f"  validation_max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}",
    "artifact_policy:",
    "  policy: slim_card_recorder_first",
    "  machine_entrypoint: summary.json",
    "",
])
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')
manifest = json.loads((PROVIDER_URI / 'qlib_store_manifest.json').read_text(encoding='utf-8'))
print(json.dumps({'provider_as_of': manifest['as_of'], 'experiment': EXPERIMENT}, indent=2, sort_keys=True))


def build_ml_throttle_frame() -> tuple[pd.DataFrame, list[str], list[str]]:
    qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
    daily_qlib_frame = D.features(
        ['SOXL', 'SOXX'],
        ['$open', '$high', '$low', '$close', '$volume', '$vix'],
        start_time=SOURCE_START,
        end_time=SOURCE_END,
        freq='day',
    )
    session_features = build_session_effect_frame(daily_qlib_frame, start=DATA_START, end=DATA_END, min_history=20)
    canary_features = build_canary_sprint_frame(daily_qlib_frame, start=DATA_START, end=DATA_END)
    components = [
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02', start=DATA_START, end=DATA_END, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=1.0, stop_loss=0.02, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03', start=DATA_START, end=DATA_END, score_name='soxx_ret252', threshold=0.15, regime_filter='vix30', scale=0.75, stop_loss=0.03, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02', start=DATA_START, end=DATA_END, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=0.75, stop_loss=0.02, lowhv_quantile=0.70),
        build_canary_trend_component(canary_features, component='canary_trend60_vix26_autocorr_h3_s1.0', start=DATA_START, end=DATA_END, trend_threshold=0.06, vix_max=26.0, hold_days=3, require_positive_autocorr=True, scale=1.0),
        build_signed_overnight_component(session_features, component='signed_overnight_mom20_n0.002_soxl60_vix45_s0.5', start=DATA_START, end=DATA_END, neutral_band=0.002, scale=0.5, momentum_window=20, trend_filter='soxl60', vix_max=45.0),
    ]
    component_frame = pd.concat(components, ignore_index=True)
    component_exposure = component_frame.pivot_table(index='datetime', columns='component', values='exposure', aggfunc='sum').sort_index().fillna(0.0)

    normalized = daily_qlib_frame.reset_index()
    normalized['instrument'] = normalized['instrument'].astype(str).str.upper()
    normalized['datetime'] = pd.to_datetime(normalized['datetime']).dt.normalize()
    soxl = normalized.loc[normalized['instrument'] == 'SOXL', ['datetime', '$open', '$high', '$low', '$close', '$volume', '$vix']].sort_values('datetime')
    soxx = normalized.loc[normalized['instrument'] == 'SOXX', ['datetime', '$close', '$volume']].sort_values('datetime').rename(columns={'$close': 'soxx_close', '$volume': 'soxx_volume'})
    panel = soxl.merge(soxx, on='datetime', how='inner').rename(columns={'$open': 'soxl_open', '$high': 'soxl_high', '$low': 'soxl_low', '$close': 'soxl_close', '$volume': 'soxl_volume', '$vix': 'vix'}).sort_values('datetime').reset_index(drop=True)
    panel['forward_return'] = panel['soxl_close'].pct_change().shift(-1)
    panel['label_return_5d'] = panel['soxl_close'].shift(-5) / panel['soxl_close'] - 1.0
    for window in [1, 2, 3, 5, 10, 20, 40, 60, 120, 126, 200, 252]:
        panel[f'soxl_ret{window}'] = panel['soxl_close'].pct_change(window)
        panel[f'soxx_ret{window}'] = panel['soxx_close'].pct_change(window)
    for window in [10, 20, 40, 60, 120, 252]:
        panel[f'soxl_hv{window}'] = panel['soxl_close'].pct_change().rolling(window, min_periods=min(window, 20)).std() * np.sqrt(252)
    for window in [20, 60, 120, 252]:
        panel[f'soxl_dd{window}'] = panel['soxl_close'] / panel['soxl_close'].rolling(window, min_periods=min(window, 20)).max() - 1.0
    panel['vix_chg1'] = panel['vix'].pct_change()
    panel['vix_chg5'] = panel['vix'].pct_change(5)
    panel['volume_ratio20'] = panel['soxl_volume'] / panel['soxl_volume'].rolling(20, min_periods=10).mean() - 1.0

    frame = component_exposure.reset_index().merge(panel, on='datetime', how='inner')
    component_cols = list(component_exposure.columns)
    old_selected_model = json.loads((ROOT / 'research' / 'agentic_alpha_ensemble_search_v1' / 'selected_model.json').read_text(encoding='utf-8'))
    old_weights = {str(key): float(value) for key, value in old_selected_model['weights'].items()}
    frame['old_raw_target'] = frame[component_cols].mul(pd.Series({column: old_weights.get(column, 0.0) for column in component_cols}), axis=1).sum(axis=1)
    frame['old_raw_cap1'] = frame['old_raw_target'].clip(0.0, 1.0)
    frame['canary_base'] = frame['canary_trend60_vix26_autocorr_h3_s1.0'].clip(0.0, 1.0)

    constrained_selected = json.loads((ROOT / 'research' / 'agentic_alpha_constrained_weight_search_v1' / 'constrained_weight_selected_model.json').read_text(encoding='utf-8'))
    _, base_positions = evaluate_weighted_position_candidates(
        frame.loc[(frame['datetime'] >= pd.Timestamp(DATA_START)) & (frame['datetime'] <= pd.Timestamp(DATA_END))],
        candidates=[{'candidate': 'constrained_base', 'weights': constrained_selected['weights'], 'policy': constrained_selected['policy']}],
        component_cols=component_cols,
        return_col='forward_return',
        cost_per_turnover=COST_PER_TURNOVER,
    )
    frame = frame.merge(base_positions[['datetime', 'target_position']].rename(columns={'target_position': 'constrained_base'}), on='datetime', how='left')
    frame['constrained_base'] = frame['constrained_base'].fillna(0.0).clip(0.0, 1.0)
    frame['base_trend_gap'] = frame['constrained_base'] - frame['old_raw_cap1']

    feature_cols = component_cols + [
        'old_raw_target', 'old_raw_cap1', 'canary_base', 'constrained_base', 'base_trend_gap',
        'soxl_ret1', 'soxl_ret2', 'soxl_ret5', 'soxl_ret10', 'soxl_ret20', 'soxl_ret60',
        'soxx_ret10', 'soxx_ret20', 'soxx_ret60', 'soxx_ret120', 'soxx_ret252',
        'soxl_hv20', 'soxl_hv60', 'soxl_dd20', 'soxl_dd60', 'soxl_dd120',
        'vix', 'vix_chg1', 'vix_chg5', 'volume_ratio20',
    ]
    frame = frame.loc[(frame['datetime'] >= pd.Timestamp(DATA_START)) & (frame['datetime'] <= pd.Timestamp(DATA_END))]
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=['forward_return']).reset_index(drop=True)
    return frame, feature_cols, component_cols

ml_frame, feature_cols, component_cols = build_ml_throttle_frame()
print(json.dumps({'rows': int(len(ml_frame)), 'features': int(len(feature_cols)), 'component_features': component_cols}, indent=2, sort_keys=True))

position_specs: list[dict[str, Any]] = []
for base_position_col in ['constrained_base', 'old_raw_cap1', 'canary_base']:
    for low_probability, high_probability in [(0.35, 0.60), (0.40, 0.65), (0.45, 0.70), (0.50, 0.75)]:
        for min_throttle in [0.35, 0.55, 0.75, 0.90]:
            for rebalance in [0.10, 0.20, 0.25]:
                for min_hold_days in [2, 3]:
                    position_specs.append({
                        'position_mode': 'base_throttle',
                        'base_position_col': base_position_col,
                        'low_probability': low_probability,
                        'high_probability': high_probability,
                        'min_throttle': min_throttle,
                        'max_throttle': 1.0,
                        'trend_floor_col': 'soxl_ret60',
                        'trend_floor_threshold': 0.0,
                        'trend_floor_fraction': max(min_throttle, 0.55),
                        'throttle_curve': 'smoothstep',
                        'scale': 1.0,
                        'rebalance_threshold': rebalance,
                        'max_daily_change': 0.50,
                        'min_hold_days': min_hold_days,
                        'min_position': 0.0,
                        'max_position': MAX_POSITION,
                        'objective_min_avg_abs_position': MIN_VALIDATION_AVG_ABS_POSITION,
                        'objective_return_weight': 0.25,
                        'objective_return_to_drawdown_weight': 1.0,
                        'objective_exposure_weight': 4.0,
                        'objective_max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
                        'turnover_penalty': 0.50,
                        'reversal_penalty': 4.0,
                        'trade_penalty': 0.05,
                    })

model_specs = [
    {'model': 'logistic_regression'},
    {'model': 'hist_gradient_boosting_classifier'},
    {'model': 'extra_trees_classifier'},
    {'model': 'random_forest_classifier'},
    {'model': 'lgbm_classifier'},
    {'model': 'ridge'},
    {'model': 'hist_gradient_boosting_regressor'},
    {'model': 'extra_trees_regressor'},
    {'model': 'random_forest_regressor'},
    {'model': 'lgbm_regressor'},
]

candidate_metrics, daily_positions = evaluate_ml_position_candidates(
    ml_frame,
    feature_cols=feature_cols,
    model_specs=model_specs,
    train_window=TRAIN_WINDOW,
    validation_window=VALIDATION_WINDOW,
    test_window=TEST_WINDOW,
    position_specs=position_specs,
    return_col=DAILY_RETURN_COL,
    label_col=LABEL_COL,
    cost_per_turnover=COST_PER_TURNOVER,
    min_validation_trade_count=MIN_VALIDATION_TRADE_COUNT,
    min_validation_avg_abs_position=MIN_VALIDATION_AVG_ABS_POSITION,
    validation_max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    max_validation_reversal_ratio=MAX_VALIDATION_REVERSAL_RATIO,
)
selected_validation_row = candidate_metrics.loc[(candidate_metrics['selected']) & (candidate_metrics['segment'] == 'validation')].iloc[0].to_dict()
selected_test_row = candidate_metrics.loc[(candidate_metrics['selected']) & (candidate_metrics['segment'] == 'test')].iloc[0].to_dict()
oracle_score_test_row = candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('selection_objective_score', ascending=False).iloc[0].to_dict()
oracle_return_test_row = candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('return_pct', ascending=False).iloc[0].to_dict()

legacy_replay = replay_soxlbacktest_reference(start=TEST_WINDOW[0], end=TEST_WINDOW[1])
legacy_metrics = legacy_replay.metrics
constrained_summary = json.loads((ROOT / 'research' / 'agentic_alpha_constrained_weight_search_v1' / 'summary.json').read_text(encoding='utf-8'))
constrained_reference = constrained_summary['selected_candidate']
raw_reference = constrained_summary['raw_position_reference']
previous_ml_summary = json.loads((ROOT / 'research' / 'agentic_alpha_ml_position_model_v1' / 'summary.json').read_text(encoding='utf-8'))
previous_ml_test = previous_ml_summary['selected_candidate']['test']

print(candidate_metrics.loc[candidate_metrics['segment'] == 'validation'].sort_values('selection_objective_score', ascending=False).head(10)[['candidate', 'return_pct', 'max_drawdown_pct', 'avg_abs_position', 'trade_count', 'next_day_reversal_count', 'next_day_reversal_ratio', 'total_turnover', 'selection_objective_score', 'selection_eligible']].to_string(index=False))
print(candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('selection_objective_score', ascending=False).head(10)[['candidate', 'return_pct', 'max_drawdown_pct', 'avg_abs_position', 'trade_count', 'next_day_reversal_count', 'next_day_reversal_ratio', 'total_turnover', 'selection_objective_score']].to_string(index=False))


def jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [jsonable(item) for item in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        numeric = float(value)
        return None if not np.isfinite(numeric) else numeric
    if pd.isna(value):
        return None
    return value


def compact_metric_row(row: dict[str, Any]) -> dict[str, Any]:
    keys = [
        'candidate', 'model', 'model_task', 'model_target_col', 'return_col', 'position_mode', 'threshold', 'scale',
        'return_pct', 'max_drawdown_pct', 'return_to_drawdown', 'selection_objective_score', 'avg_abs_position',
        'max_position', 'min_position', 'days_above_100', 'days_below_0', 'total_turnover', 'trade_count',
        'next_day_reversal_count', 'next_day_reversal_ratio', 'avg_holding_days', 'selection_eligible',
        'selection_gate', 'params',
    ]
    return {key: jsonable(row.get(key)) for key in keys if key in row}

candidate_metrics_path = CARD / 'ml_throttle_candidate_metrics.csv'
selected_model_path = CARD / 'ml_throttle_selected_model.json'
ml_summary_path = CARD / 'ml_throttle_summary.json'
benchmark_path = CARD / 'backtest_benchmark_comparison.csv'
failure_modes_path = CARD / 'failure_modes.md'
daily_positions_tmp = CARD / '.ml_throttle_daily_positions.csv.gz'
candidate_metrics_full_tmp = CARD / '.ml_throttle_candidate_metrics_full.csv.gz'

selected_names_for_slim = set(candidate_metrics.loc[candidate_metrics['selected'], 'candidate'].astype(str))
slim_candidate_metrics = pd.concat([
    candidate_metrics.loc[candidate_metrics['selected']],
    candidate_metrics.loc[candidate_metrics['segment'] == 'validation'].sort_values('selection_objective_score', ascending=False).head(80),
    candidate_metrics.loc[candidate_metrics['segment'] == 'validation'].sort_values('return_pct', ascending=False).head(80),
    candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('selection_objective_score', ascending=False).head(80),
    candidate_metrics.loc[candidate_metrics['segment'] == 'test'].sort_values('return_pct', ascending=False).head(80),
], ignore_index=True).drop_duplicates(subset=['candidate', 'segment']).reset_index(drop=True)
slim_candidate_metrics.to_csv(candidate_metrics_path, index=False)
candidate_metrics.to_csv(candidate_metrics_full_tmp, index=False, compression='gzip')
daily_positions.to_csv(daily_positions_tmp, index=False, compression='gzip')
selected_model_payload = {
    'candidate': selected_test_row['candidate'],
    'selected_by': 'validation_selection_objective_score',
    'validation': compact_metric_row(selected_validation_row),
    'test': compact_metric_row(selected_test_row),
}
selected_model_path.write_text(json.dumps(selected_model_payload, indent=2, sort_keys=True), encoding='utf-8')

benchmark = pd.DataFrame([
    {'benchmark': 'selected_ml_throttle_validation_model', 'return_pct': float(selected_test_row['return_pct']), 'max_drawdown_pct': float(selected_test_row['max_drawdown_pct']), 'max_position': float(selected_test_row['max_position']), 'avg_abs_position': float(selected_test_row['avg_abs_position']), 'trade_count': int(selected_test_row['trade_count']), 'next_day_reversal_count': int(selected_test_row['next_day_reversal_count'])},
    {'benchmark': 'previous_ml_position_validation_model', 'return_pct': float(previous_ml_test['return_pct']), 'max_drawdown_pct': float(previous_ml_test['max_drawdown_pct']), 'max_position': float(previous_ml_test['max_position']), 'avg_abs_position': float(previous_ml_test['avg_abs_position']), 'trade_count': int(previous_ml_test['trade_count']), 'next_day_reversal_count': int(previous_ml_test['next_day_reversal_count'])},
    {'benchmark': 'oracle_score_test_best_ml_throttle_candidate', 'return_pct': float(oracle_score_test_row['return_pct']), 'max_drawdown_pct': float(oracle_score_test_row['max_drawdown_pct']), 'max_position': float(oracle_score_test_row['max_position']), 'avg_abs_position': float(oracle_score_test_row['avg_abs_position']), 'trade_count': int(oracle_score_test_row['trade_count']), 'next_day_reversal_count': int(oracle_score_test_row['next_day_reversal_count'])},
    {'benchmark': 'oracle_return_test_best_ml_throttle_candidate', 'return_pct': float(oracle_return_test_row['return_pct']), 'max_drawdown_pct': float(oracle_return_test_row['max_drawdown_pct']), 'max_position': float(oracle_return_test_row['max_position']), 'avg_abs_position': float(oracle_return_test_row['avg_abs_position']), 'trade_count': int(oracle_return_test_row['trade_count']), 'next_day_reversal_count': int(oracle_return_test_row['next_day_reversal_count'])},
    {'benchmark': 'constrained_weight_reference', 'return_pct': float(constrained_reference['return_pct']), 'max_drawdown_pct': float(constrained_reference['max_drawdown_pct']), 'max_position': float(constrained_reference['max_position']), 'avg_abs_position': None, 'trade_count': int(constrained_reference['trade_count']), 'next_day_reversal_count': int(constrained_reference['next_day_reversal_count'])},
    {'benchmark': 'raw_target_position_reference', 'return_pct': float(raw_reference['return_pct']), 'max_drawdown_pct': float(raw_reference['max_drawdown_pct']), 'max_position': float(raw_reference['max_position']), 'avg_abs_position': float(raw_reference['avg_abs_position']), 'trade_count': int(raw_reference['trade_count']), 'next_day_reversal_count': int(raw_reference['next_day_reversal_count'])},
    {'benchmark': 'soxl_backtest_reference', 'return_pct': float(legacy_metrics['rulebase_return_pct']), 'max_drawdown_pct': -float(legacy_metrics['rulebase_max_drawdown_pct']), 'max_position': float(legacy_replay.daily['position'].max()), 'avg_abs_position': float(legacy_replay.daily['position'].abs().mean()), 'trade_count': int(legacy_metrics['trade_count']), 'next_day_reversal_count': None},
    {'benchmark': 'soxl_buyhold', 'return_pct': float(legacy_metrics['soxl_buyhold_return_pct']), 'max_drawdown_pct': None, 'max_position': 1.0, 'avg_abs_position': 1.0, 'trade_count': 1, 'next_day_reversal_count': None},
])
benchmark.to_csv(benchmark_path, index=False)
model_families = ['ExtraTreesClassifier', 'ExtraTreesRegressor', 'HistGradientBoostingClassifier', 'HistGradientBoostingRegressor', 'LGBMClassifier', 'LGBMRegressor', 'LogisticRegression', 'RandomForestClassifier', 'RandomForestRegressor', 'Ridge']
beats_previous_ml = float(selected_test_row['return_pct']) > float(previous_ml_test['return_pct'])
beats_soxlbacktest = float(selected_test_row['return_pct']) > float(legacy_metrics['rulebase_return_pct'])
beats_constrained = float(selected_test_row['return_pct']) > float(constrained_reference['return_pct'])
ml_throttle_summary = {
    'workflow': EXPERIMENT,
    'goal_achieved': bool(beats_previous_ml and beats_soxlbacktest and beats_constrained),
    'candidate_count': int(candidate_metrics['candidate'].nunique()),
    'metric_rows': int(len(candidate_metrics)),
    'feature_count': int(len(feature_cols)),
    'feature_cols': feature_cols,
    'component_cols': component_cols,
    'model_families': model_families,
    'selection_gate': {'selection_segment': 'validation', 'report_segment': 'test', 'model_target_col': LABEL_COL, 'return_col': DAILY_RETURN_COL, 'min_validation_trade_count': MIN_VALIDATION_TRADE_COUNT, 'min_validation_avg_abs_position': MIN_VALIDATION_AVG_ABS_POSITION, 'max_validation_reversal_ratio': MAX_VALIDATION_REVERSAL_RATIO, 'validation_max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'max_position': MAX_POSITION},
    'selected_candidate': selected_model_payload,
    'oracle_score_test_best_candidate': compact_metric_row(oracle_score_test_row),
    'oracle_return_test_best_candidate': compact_metric_row(oracle_return_test_row),
    'previous_ml_position_reference': previous_ml_test,
    'constrained_weight_reference': constrained_reference,
    'raw_position_reference': raw_reference,
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'comparison': {'selected_vs_previous_ml_return_gap_pct': float(selected_test_row['return_pct'] - previous_ml_test['return_pct']), 'selected_vs_soxlbacktest_return_gap_pct': float(selected_test_row['return_pct'] - legacy_metrics['rulebase_return_pct']), 'selected_vs_constrained_return_gap_pct': float(selected_test_row['return_pct'] - constrained_reference['return_pct']), 'beats_previous_ml': bool(beats_previous_ml), 'beats_soxlbacktest': bool(beats_soxlbacktest), 'beats_constrained_weight_reference': bool(beats_constrained)},
}
ml_summary_path.write_text(json.dumps(jsonable(ml_throttle_summary), indent=2, sort_keys=True), encoding='utf-8')
failure_modes_path.write_text('# Failure Modes\n\n- ML throttle keeps the base prior and controls over-100% exposure, but it can still underperform if the validated base spends the trend leg fully invested.\n- Selection is validation-only; oracle test rows are diagnostic and are not promotion evidence.\n- The model learns 5-day labels while the report uses daily PnL, reducing next-day churn but not eliminating sample-size risk.\n- Single-asset daily ML remains fragile; production use still requires paper-sim and walk-forward refresh.\n', encoding='utf-8')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(provider_uri=PROVIDER_URI.as_posix(), train_start=TRAIN_WINDOW[0], train_end=TRAIN_WINDOW[1], validation_start=VALIDATION_WINDOW[0], validation_end=VALIDATION_WINDOW[1], test_start=TEST_WINDOW[0], test_end=TEST_WINDOW[1], feature_count=int(len(feature_cols)), candidate_count=int(candidate_metrics['candidate'].nunique()), min_validation_trade_count=MIN_VALIDATION_TRADE_COUNT, min_validation_avg_abs_position=MIN_VALIDATION_AVG_ABS_POSITION, max_validation_reversal_ratio=MAX_VALIDATION_REVERSAL_RATIO, validation_max_drawdown_floor=MAX_DRAWDOWN_FLOOR, cost_per_turnover=COST_PER_TURNOVER)
    recorder.log_metrics(selected_test_return_pct=float(selected_test_row['return_pct']), selected_test_max_drawdown_pct=float(selected_test_row['max_drawdown_pct']), selected_test_avg_abs_position=float(selected_test_row['avg_abs_position']), selected_test_reversal_ratio=float(selected_test_row['next_day_reversal_ratio']), previous_ml_return_pct=float(previous_ml_test['return_pct']), constrained_weight_return_pct=float(constrained_reference['return_pct']), soxlbacktest_return_pct=float(legacy_metrics['rulebase_return_pct']))
    for path in [candidate_metrics_path, selected_model_path, ml_summary_path, benchmark_path, failure_modes_path]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_ml_throttle')
    recorder.log_artifact(daily_positions_tmp.as_posix(), artifact_path='agentic_alpha_ml_throttle')
    recorder.log_artifact(candidate_metrics_full_tmp.as_posix(), artifact_path='agentic_alpha_ml_throttle')
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.uri).resolve()

daily_positions_tmp.unlink(missing_ok=True)
candidate_metrics_full_tmp.unlink(missing_ok=True)
ml_throttle_summary['recorder_id'] = recorder_id
ml_throttle_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
ml_throttle_summary['recorder_artifacts'] = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
ml_summary_path.write_text(json.dumps(jsonable(ml_throttle_summary), indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps({'recorder_id': recorder_id, 'selected_test_return_pct': float(selected_test_row['return_pct']), 'beats_soxlbacktest': bool(beats_soxlbacktest), 'beats_constrained': bool(beats_constrained)}, indent=2, sort_keys=True))

artifact_policy = {
    'policy': 'slim_card_recorder_first',
    'workflow_config': 'workflow.yaml',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'kept_sidecars': ['backtest_benchmark_comparison.csv', 'ml_throttle_candidate_metrics.csv', 'ml_throttle_selected_model.json', 'ml_throttle_summary.json', 'failure_modes.md'],
    'recorder_only_artifacts': {'predictions': 'artifacts/pred.pkl', 'labels': 'artifacts/label.pkl', 'signals': 'artifacts/signals.pkl', 'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv', 'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl', 'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl', 'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl', 'daily_positions': 'artifacts/agentic_alpha_ml_throttle/.ml_throttle_daily_positions.csv.gz', 'candidate_metrics_full': 'artifacts/agentic_alpha_ml_throttle/.ml_throttle_candidate_metrics_full.csv.gz'},
}
promotion_decision = 'promote_to_production_review' if beats_previous_ml and beats_soxlbacktest and beats_constrained else 'not_promoted_ml_throttle_under_constrained_weight'
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'ml_throttle_position_search',
    'promotion_decision': promotion_decision,
    'as_of': DATA_END,
    'market': 'all',
    'region': 'us',
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'data_range': [DATA_START, DATA_END],
    'segments': {'train': list(TRAIN_WINDOW), 'validation': list(VALIDATION_WINDOW), 'test': list(TEST_WINDOW)},
    'goal_gate': {'selection_segment': 'validation', 'report_segment': 'test', 'model_target_col': LABEL_COL, 'return_col': DAILY_RETURN_COL, 'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'max_position': MAX_POSITION, 'min_validation_trade_count': MIN_VALIDATION_TRADE_COUNT, 'min_validation_avg_abs_position': MIN_VALIDATION_AVG_ABS_POSITION, 'max_validation_reversal_ratio': MAX_VALIDATION_REVERSAL_RATIO, 'require_return_above_previous_ml': True, 'require_return_above_soxlbacktest': True, 'require_return_above_constrained_weight_reference': True},
    'selected_candidate': selected_model_payload,
    'oracle_score_test_best_candidate': compact_metric_row(oracle_score_test_row),
    'oracle_return_test_best_candidate': compact_metric_row(oracle_return_test_row),
    'previous_ml_position_reference': previous_ml_test,
    'constrained_weight_reference': constrained_reference,
    'raw_position_reference': raw_reference,
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'ml_throttle_report': ml_throttle_summary,
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'qlib_docs': {'data': 'https://qlib.readthedocs.io/en/latest/component/data.html', 'model': 'https://qlib.readthedocs.io/en/latest/component/model.html', 'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html'},
    'workflow_config': 'workflow.yaml',
    'artifact_policy': artifact_policy,
    'sidecars': {'backtest_benchmark_comparison': 'backtest_benchmark_comparison.csv', 'candidate_metrics': 'ml_throttle_candidate_metrics.csv', 'failure_modes': 'failure_modes.md', 'ml_throttle_summary': 'ml_throttle_summary.json', 'selected_model': 'ml_throttle_selected_model.json'},
    'recorder_artifacts': ml_throttle_summary['recorder_artifacts'],
}
(CARD / 'summary.json').write_text(json.dumps(jsonable(summary), indent=2, sort_keys=True), encoding='utf-8')
conclusion = (
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，用 qlib.init + D.features 构建 SOXL/SOXX 日线特征、agentic component exposure、constrained_base/old_raw/canary base prior，并训练 Logistic/Tree/LightGBM/Ridge 系列 ML throttle。\n'
    f'- key_metrics: validation 选中 {selected_model_payload["candidate"]}；2026 test return={selected_model_payload["test"]["return_pct"]:.4f}%，MaxDD={selected_model_payload["test"]["max_drawdown_pct"]:.4f}%，avg_abs_position={selected_model_payload["test"]["avg_abs_position"]:.4f}，trade_count={selected_model_payload["test"]["trade_count"]}，next_day_reversal_ratio={selected_model_payload["test"]["next_day_reversal_ratio"]:.4f}。\n'
    f'- benchmark: previous ML return={previous_ml_test["return_pct"]:.4f}%，soxl_backtest return={legacy_metrics["rulebase_return_pct"]:.4f}%，constrained_weight_reference return={constrained_reference["return_pct"]:.4f}%，oracle_return_test_top={oracle_return_test_row["return_pct"]:.4f}%。\n'
    '- regime/window differences: train=2019-2024，validation=2025，test=2026-01-02 到 2026-06-09；模型学习 5 日 label，但回放用 daily PnL。新架构把仓位上限和换手惯性放在 policy 中，ML 只缩放 base prior。\n'
    f'- promotion_decision: {promotion_decision}。若没有超过 constrained_weight_reference，则只作为 ML overlay 诊断，不替代当前约束权重模型。\n'
    '- risks: 单标的日线 ML 样本仍小；constrained_base 本身已经很强，throttle 可能降低趋势期收益；oracle test-top 仍不能当作生产证据。\n'
    f'- next_action: 保留为研究卡，后续若继续 ML，应做 rolling walk-forward/online refit 和 production paper-sim。qlib recorder id={recorder_id}。\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[56707:MainThread](2026-06-24 21:31:08,248) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56707:MainThread](2026-06-24 21:31:08,250) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56707:MainThread](2026-06-24 21:31:08,251) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{
  "experiment": "agentic_alpha_ml_throttle_search_v1",
  "provider_as_of": "2026-06-23"
}
{
  "component_features": [
    "canary_trend60_vix26_autocorr_h3_s1.0",
    "signed_overnight_mom20_n0.002_soxl60_vix45_s0.5",
    "tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02",
    "tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02",
    "tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03"
  ],
  "features": 30,
  "rows": 1869
}


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[56707:MainThread](2026-06-24 21:31:34,573) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_ml_throttle_search_v1.


[56707:MainThread](2026-06-24 21:31:34,577) INFO - qlib.workflow - [exp.py:258] - Experiment 955025840187773926 starts running ...


[56707:MainThread](2026-06-24 21:31:34,699) INFO - qlib.workflow - [recorder.py:345] - Recorder 3d415e61f8a94d278ab49f7fe2068d16 starts running under Experiment 955025840187773926 ...


[56707:MainThread](2026-06-24 21:31:34,819) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{
  "beats_constrained": false,
  "beats_soxlbacktest": true,
  "recorder_id": "3d415e61f8a94d278ab49f7fe2068d16",
  "selected_test_return_pct": 182.33748999142975
}
# 结论

- what_ran: 执行 agentic_alpha_ml_throttle_search_v1，用 qlib.init + D.features 构建 SOXL/SOXX 日线特征、agentic component exposure、constrained_base/old_raw/canary base prior，并训练 Logistic/Tree/LightGBM/Ridge 系列 ML throttle。
- key_metrics: validation 选中 extra_trees_classifier_base_throttle_constrained_base_p0.4-0.65_t0.75-1_s1_h0.25_hold3；2026 test return=182.3375%，MaxDD=-9.3259%，avg_abs_position=0.3163，trade_count=13，next_day_reversal_ratio=0.0000。
- benchmark: previous ML return=17.1388%，soxl_backtest return=132.0780%，constrained_weight_reference return=571.2201%，oracle_return_test_top=287.8333%。
- regime/window differences: train=2019-2024，validation=2025，test=2026-01-02 到 2026-06-09；模型学习 5 日 label，但回放用 daily PnL。新架构把仓位上限和换手惯性放在 policy 中，ML 只缩放 base prior。
- promotion_decision: not_promoted_ml_throttle_under_constrained_weigh